In [1]:
import pandas as pd
import os
from pathlib import Path
import sys
import time
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import meteostat
from tqdm import tqdm
import pickle

In [2]:
#get list of cities
df_cities = pd.read_csv('/home/kieran/.cache/huggingface/hub/datasets--NUS-UAL--global-streetscapes/snapshots/7bd2e7697a3cb5f74ff05bd718babdb927f8b60d/cities688.csv')


In [ ]:
def hotDayFinder(city_ascii,city_lat,city_lon,df_streetscapes):
    flag = 0
    start = datetime(2018,1,1)
    end = datetime(2025,1,1)
    #print(city_ascii)
    city_point = meteostat.Point(city_lat,city_lon,0,alt_range = 2000) #altitude of 0 since idk
    #print(city_point)
    try:
        data = meteostat.Daily(city_point,start,end) #get data for stuff
    except:
        print('temp data failed')
        return 0,flag
    data = data.fetch()#ping a req for data
    hot_days = data[data['tmax'] >= 34] #arbitrary max setting, could be based on stdev of location? Also in C (assumed to be standard across all countries?)
    if len(hot_days) <1:
        print('no hot days')
        flag = 1
        return 0,flag
    hot_days_list = hot_days.index.date #list of hot dates?
    hot_days_set = set(hot_days_list)
    df_city = df_streetscapes[df_streetscapes['city_ascii']== city_ascii]
    
    img_dates = pd.to_datetime(df_city['datetime_local'],format = 'ISO8601',utc = True) #utc is actually not true, preventing an error?
    #print(img_dates)
    img_date_set = set(img_dates.dt.date)#making it into a set guarantees unique vals (no dupes). Turning it into dates makes stuff only days, no mins/sec
    intersecting_days = list(hot_days_set.intersection(img_date_set))
    if len(intersecting_days) <1:
        print('no intersecting days')
        flag = 1
        return 0,flag
    return intersecting_days,flag

In [4]:
df_streetscapes = pd.read_csv(
    "/home/kieran/.cache/huggingface/hub/datasets--NUS-UAL--global-streetscapes/snapshots/7bd2e7697a3cb5f74ff05bd718babdb927f8b60d/data/simplemaps.csv"
)

In [5]:
#add in another dataset to get the year of the image -> check for covid?
df_metadata = pd.read_csv('/home/kieran/.cache/huggingface/hub/datasets--NUS-UAL--global-streetscapes/snapshots/7bd2e7697a3cb5f74ff05bd718babdb927f8b60d/data/metadata_common_attributes.csv')
df_streetscapes = df_streetscapes.merge(df_metadata, on = ["uuid","source","orig_id"])
df_metadata = None

In [9]:
Urasoe_row = df_cities.iloc[0]
test,temp = hotDayFinder('Urasoe',Urasoe_row['city_lat'],Urasoe_row['city_lon'],df_streetscapes)

no intersecting days


In [10]:
unique_cities = df_streetscapes['city_ascii'].unique()
#unique_cities = unique_cities[10:20]
hot_days = []
hot_cities = []
for city in tqdm(unique_cities):
    city_lat = df_cities[df_cities['city_ascii'] == city]['city_lat']
    city_lon = df_cities[df_cities['city_ascii']== city]['city_lon']
    city_lat = city_lat.values[0]
    city_lon = city_lon.values[0]
    #print(city_lat)
    #print(city_lon)
    temp_output, flag = hotDayFinder(city,city_lat,city_lon,df_streetscapes)
    if flag ==0:
        hot_days.append(temp_output)
        hot_cities.append(city)

  0%|          | 1/684 [00:00<04:27,  2.55it/s]

no intersecting days
temp data failed


  0%|          | 3/684 [00:00<02:43,  4.17it/s]

no intersecting days


  1%|          | 4/684 [00:01<03:11,  3.55it/s]

...

temp data failed


100%|█████████▉| 683/684 [17:49<00:04,  4.05s/it]

no intersecting days


100%|██████████| 684/684 [17:50<00:00,  1.56s/it]

no intersecting days


In [ ]:
filepath = 'test_day_list_2.pkl'
with open(filepath,'wb') as f:
    pickle.dump(hot_days,f)
filepath = 'test_city_list_2.pkl'
with open(filepath,'wb') as f:
    pickle.dump(hot_cities,f)

In [27]:
print(type(hot_days[3][0]))

<class 'datetime.date'>


In [18]:
print((hot_cities))

['Sasagawa', 'Betim', 'Sirsilla', 'Kasuga', 'Caxias do Sul', 'Provo', 'Awasa', 'Monterrey', 'Downey', 'Chililabombwe', 'El Cajon', 'Poprad', 'Ebina', 'Moscow', 'Olathe', 'Aleksin', 'Burnley', 'Holguin', 'Gilbert', 'La Dorada', 'Melbourne', 'Saarbrucken', 'Bombo', 'Volta Redonda', 'Chandigarh', 'Vizianagaram', 'Beckley', 'Rennes', 'Airdrie', 'Crato', 'Itumbiara', 'Vienna', 'Bentonville', 'Bakersfield', 'Southfield', 'Madrid', 'Pisa', 'Chennai', 'Brandon', 'Ueda', "Cava de' Tirreni", 'Niteroi', 'Philadelphia', 'Zurich', 'Hanau', 'Santa Isabel', 'Chuo-ku', 'Thane', 'Saint-Louis', 'Rio do Sul', 'Cainta', 'San Antonio', 'Los Andes', 'Blumenau', 'Pekalongan', 'Maipu', 'Frisco', 'Orleans', 'Wausau', 'Vereeniging', 'Pocos de Caldas', 'Vyronas', 'Cahama', 'Chongshan', 'Kansas City', 'Rodriguez', 'Halabjah', 'Parla', 'San Jose del Monte', 'Tagaytay', 'Can Tho', 'Narutocho-mitsuishi', 'Almirante Tamandare', 'Long Beach', 'Livermore', 'San Cristobal', 'Getafe', 'Temple', 'Ravensburg', 'Goslar', 'R

In [21]:
#now take hot cities/days, find list of SVI from those days
df_streetscapes['date_standardized'] = pd.to_datetime(df_streetscapes['datetime_local'],format = 'ISO8601',utc = True).dt.date #convert to dates? idk if this will work

In [23]:
df_streetscapes['date_standardized'].iloc[0]

datetime.date(2021, 11, 3)

In [28]:
svi_df_list = []
for i in range(0,len(hot_days)):
    if hot_days[i] == 0:
        pass
    else:
        city_ascii = hot_cities[i]
        df_city = df_streetscapes[df_streetscapes['city_ascii'] == city_ascii]
        df_city_day = df_city[df_city['date_standardized'] == hot_days[i][0] ]#index into hot days to make it a single var -> assumes that there is one hot day per city
        svi_df_list.append(df_city_day)

In [36]:
#concat df to make one mega manifest 
df_hot_images = pd.concat(svi_df_list, axis = 0)
print(len(df_hot_images))

56809


In [5]:
#load in hot imgs from csv
df_hot_images = pd.read_csv('/home/kieran/Documents/Python/sunny_day_SVI/hot_image_manifest.csv')

In [4]:
#now apply filtering
#filter based on walkable street (to not just get roads). Technically optional. Could try without if necessary
df_osm = pd.read_csv('/home/kieran/.cache/huggingface/hub/datasets--NUS-UAL--global-streetscapes/snapshots/7bd2e7697a3cb5f74ff05bd718babdb927f8b60d/data/osm.csv')
df_hot_images = df_hot_images.merge(df_osm, on = ["uuid","source","orig_id"])
allowed_street_types = ['walk','walk/cycle','cycle','others']
df_hot_images = df_hot_images[df_hot_images['type_highway'].isin(allowed_street_types)]
df_osm = None

KeyError: 'type_highway'

In [6]:
#add data for num people in the img
#add in another dataset to get the year of the image -> check for covid?
df_instances = pd.read_csv('/home/kieran/.cache/huggingface/hub/datasets--NUS-UAL--global-streetscapes/snapshots/7bd2e7697a3cb5f74ff05bd718babdb927f8b60d/data/instances.csv')
df_instances = df_instances[['uuid','source','orig_id',"Person"]]#could collect more things if necessary
df_hot_images = df_hot_images.merge(df_instances, on = ["uuid","source","orig_id"])
df_instances = None

In [7]:
#minor checks on the dataframe
print(len(df_hot_images['city'].unique()))

101


In [8]:
print(df_hot_images['Person'].sum()/len(df_hot_images))

2.4016929588303193


In [9]:
print(df_hot_images['Person'].sum())

18726.0


In [2]:
#save list of images
csv_dir = 'hot_image_manifest.csv'
df_hot_images.to_csv(csv_dir)

NameError: name 'df_hot_images' is not defined

In [ ]:
#Append sys path to allow import (need to change to be more general)
sys.path.append('/home/kieran/Documents/Datasets/Global streetscapes/global-streetscapes/code/download_imgs') #hardcoded path for images

#import download script from relevant file (copy of the manual version)
from download_jpegs import download_csv


In [45]:
folderpath = 'hot_imgs_whole'
#execute download
download_csv(csv_dir,folderpath)

Checking all subfolders for existing images...
Found 0 images in hot_imgs_whole
Initiating download for new images...
Now: 100 / 7918 . Pre-existing: 0
Requesting GET to https://graph.mapillary.com/921316432054049/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/302703011481510/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/286011913165010/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/514180566418225/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/158746416187102/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/369143204515057/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/504352407375484/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/564947038244150/?fields=thumb_2048_url,geometry
Requesting GET to https://graph.mapillary.com/317363580048697/?fields=thumb_2048_url,geometry
Re